# EquipAnalytics — análise guiada
Dados sintéticos. Inicie o Jupyter a partir de `projetos/big-data-equipamentos` ou da pasta notebooks. Execute antes o pipeline com saída `data/run-100k`. Este notebook não contém saídas pré-executadas.


In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'pipeline.py').exists(), 'Inicie na pasta do projeto'
spark = (SparkSession.builder.master('local[2]').appName('EquipAnalyticsAnalysis')
         .config('spark.sql.session.timeZone', 'UTC').getOrCreate())
RUN = ROOT / 'data/run-100k'
silver = spark.read.parquet(str(RUN / 'silver/events'))
silver.createOrReplaceTempView('silver_events')


## Onde investigar primeiro?
O ranking combina quantidade de alertas e custo; ele não diagnostica a causa de consumo elevado.


In [ ]:
ranking = spark.sql((ROOT / 'sql/equipment_alerts.sql').read_text())
ranking.show(20, truncate=False)


In [ ]:
monthly = spark.sql((ROOT / 'sql/monthly_kpis.sql').read_text())
monthly.show(20, truncate=False)


## Qualidade e desempenho
Confira reconciliação e motivos; examine o plano de leitura para identificar filtros de partição. Duração local não comprova ganho em cluster.


In [ ]:
import json
metrics = json.loads((RUN / 'metrics.json').read_text())
assert metrics['bronze_rows'] == metrics['silver_rows'] + metrics['rejected_rows'] + metrics['duplicate_rows']
metrics


In [ ]:
silver.where("year_month = '2025-01'").groupBy('unit').sum('liters').explain(mode='formatted')


## Registrar conclusões
Após executar, descreva: unidades com maior consumo, equipamentos a investigar, qualidade da entrada e limitações do contexto. Não conclua fraude, economia ou falha mecânica apenas com estes dados.


In [ ]:
spark.stop()
